<a href="https://colab.research.google.com/github/NicGal-1925/streaming-recommendation-system/blob/main/notebooks/04_preprocessing_and_target.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocessing, Target Definition, and Temporal Split

## Purpose

Prepare MovieLens ml-latest-small interactions for positive-preference
prediction. Each row represents one user rating one movie.

This notebook validates the source data, converts timestamps to UTC,
multi-hot encodes movie genres, defines the binary target, and creates
a per-user chronological training/test split.

## Target and Split

- `positive_rating = 1` when the original rating is 4.0 or higher;
  otherwise `0`.
- Each user's earliest approximately 80% of interactions form training.
- The remaining approximately 20% form testing.
- Ratings sharing the same recorded second may cross the split boundary.
  This timestamp-resolution limitation is checked and documented below.

## Output and Notebook Responsibilities

The output is `movielens_prepared_split.zip`, containing
`train_data.csv` and `test_data.csv`.

These tables retain ratings, identifiers, timestamps, genre indicators,
the target, and split bookkeeping. Retaining a column does not mean it
is an approved model input: the original rating directly reveals the
target, and split bookkeeping must not be used as predictive features.

Notebook 05 selects the model inputs, trains the movie-genre baseline,
and evaluates its predictions. Later experiments must reuse the exact
same held-out test interactions.

## Execution

Run the cells from top to bottom in a fresh Colab runtime.
The source data is downloaded from GroupLens. Keep the exported archive
to preserve the exact prepared split used by downstream experiments.

In [1]:
import pandas as pd
import zipfile
import urllib.request
import os

# download the public MovieLens ZIP directly into this Colab runtime.
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zip_path = "/content/ml-latest-small.zip"

urllib.request.urlretrieve(url, zip_path)

# extract it
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content/data")

# load the four DataFrames
ratings = pd.read_csv("/content/data/ml-latest-small/ratings.csv")
movies = pd.read_csv("/content/data/ml-latest-small/movies.csv")
tags = pd.read_csv("/content/data/ml-latest-small/tags.csv")
links = pd.read_csv("/content/data/ml-latest-small/links.csv")

print("Ratings:", ratings.shape)
print("Movies:", movies.shape)
print("Tags:", tags.shape)
print("Links:", links.shape)

Ratings: (100836, 4)
Movies: (9742, 3)
Tags: (3683, 4)
Links: (9742, 3)


## 1. Data Quality Validation

The datasets were checked for missing values, exact duplicates, and logical-key duplicates. No duplicate records were found. Missingness was limited to eight tmdbId values in the links table, which were retained because TMDB identifiers are not required by the current prediction pipeline.

In [2]:
# create the Python dictionary
dataframes = {
    "ratings": ratings,
    "movies": movies,
    "tags": tags,
    "links": links
}

# Check for missing values

for name, df in dataframes.items():
    print(f"\n{name.upper()}")
    print(df.isnull().sum()) # Count how many missing values exist in each column.


RATINGS
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

MOVIES
movieId    0
title      0
genres     0
dtype: int64

TAGS
userId       0
movieId      0
tag          0
timestamp    0
dtype: int64

LINKS
movieId    0
imdbId     0
tmdbId     8
dtype: int64


In [3]:
# Check for exact duplicate rows in each dataframe

for name, df in dataframes.items():
    print(f"{name}: {df.duplicated().sum()} exact duplicate rows")

ratings: 0 exact duplicate rows
movies: 0 exact duplicate rows
tags: 0 exact duplicate rows
links: 0 exact duplicate rows


In [4]:
# Compare duplicate rows

print(
    "Repeated user/movie rating pairs:",
    ratings.duplicated(subset=["userId", "movieId"]).sum() # Ignore the rating and timestamp for a moment. Has this same user/movie combination appeared before?
)                                                          # Check for duplicates according to a 'logical key'

print(
    "Repeated movie IDs in movies:",
    movies.duplicated(subset=["movieId"]).sum()
)

print(
    "Repeated movie IDs in links:",
    links.duplicated(subset=["movieId"]).sum()
)

print(
    "Repeated user/movie/tag combinations:",
    tags.duplicated(
        subset=["userId", "movieId", "tag"]
    ).sum()
)

Repeated user/movie rating pairs: 0
Repeated movie IDs in movies: 0
Repeated movie IDs in links: 0
Repeated user/movie/tag combinations: 0


## 2. Check Data Types

In [5]:
# Inspect the columns and their data types in this notebook.

for name, df in dataframes.items(): # dataframes is the name of the dictionary
    print(f"\n{name.upper()}")
    print(df.dtypes)


RATINGS
userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object

MOVIES
movieId     int64
title      object
genres     object
dtype: object

TAGS
userId        int64
movieId       int64
tag          object
timestamp     int64
dtype: object

LINKS
movieId      int64
imdbId       int64
tmdbId     float64
dtype: object


### 2.1 Convert Timestamps

In [6]:
# Create new column

ratings["rating_datetime"] = pd.to_datetime(
    ratings["timestamp"],
    unit="s",
    utc=True
)

print(ratings[["timestamp", "rating_datetime"]].head())

   timestamp           rating_datetime
0  964982703 2000-07-30 18:45:03+00:00
1  964981247 2000-07-30 18:20:47+00:00
2  964982224 2000-07-30 18:37:04+00:00
3  964983815 2000-07-30 19:03:35+00:00
4  964982931 2000-07-30 18:48:51+00:00


In [7]:
# Check the historical span of the dataset

print("Earliest rating:", ratings["rating_datetime"].min())
print("Latest rating:", ratings["rating_datetime"].max())

Earliest rating: 1996-03-29 18:36:55+00:00
Latest rating: 2018-09-24 14:27:30+00:00


### 2.2 Encode Movie Genres

Transform the pipe-separated genre labels into numerical binary features so the model can use each genre individually.

In [8]:
# see movies together with their genre combination

print(movies[["title", "genres"]].head(10))

# What distinct genre strings exist in this column?

all_genres = movies["genres"].str.split("|").explode()

print(all_genres.value_counts())

                                title  \
0                    Toy Story (1995)   
1                      Jumanji (1995)   
2             Grumpier Old Men (1995)   
3            Waiting to Exhale (1995)   
4  Father of the Bride Part II (1995)   
5                         Heat (1995)   
6                      Sabrina (1995)   
7                 Tom and Huck (1995)   
8                 Sudden Death (1995)   
9                    GoldenEye (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
5                        Action|Crime|Thriller  
6                               Comedy|Romance  
7                           Adventure|Children  
8                                       Action  
9                    Action|Adventure|Thriller  
genres
Dr

In [9]:
genre_features = movies["genres"].str.get_dummies(sep="|")

genre_features.head()

,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [10]:
genre_features.shape

(9742, 20)

### 2.3 Connect Genre Features to Movie IDs

Attach each movie’s movieId to its encoded genre features so the features can later be merged with rating data.

In [11]:
movie_features = pd.concat(
    [movies[["movieId"]], genre_features],
    axis=1
)

movie_features.head()

,movieId,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,0,0,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,0,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,0,0,0,0,0,1,0,0,1,...,0,0,0,0,0,1,0,0,0,0
4,5,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 3. Build the Modeling Dataset

Combine user-rating interactions with movie features using movieId as the common key.

In [12]:
model_data = ratings.merge(
    movie_features,
    on="movieId",
    how="left"
)

model_data.head()

,userId,movieId,rating,timestamp,rating_datetime,(no genres listed),Action,Adventure,Animation,Children,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,1,4.0,964982703,2000-07-30 18:45:03+00:00,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,1,3,4.0,964981247,2000-07-30 18:20:47+00:00,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,1,6,4.0,964982224,2000-07-30 18:37:04+00:00,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,1,47,5.0,964983815,2000-07-30 19:03:35+00:00,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
4,1,50,5.0,964982931,2000-07-30 18:48:51+00:00,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0


In [13]:
print("Ratings shape:", ratings.shape)
print("Model data shape:", model_data.shape)

Ratings shape: (100836, 5)
Model data shape: (100836, 25)


## 4. Define the Classification Target

Convert the numerical rating into a binary target indicating whether the user expressed a positive preference for the movie.

In [14]:
model_data["positive_rating"] = (
    model_data["rating"] >= 4.0
).astype(int)

In [15]:
print(model_data["positive_rating"].value_counts())

print(
    model_data["positive_rating"]
    .value_counts(normalize=True)
    .mul(100)
)

positive_rating
0    52256
1    48580
Name: count, dtype: int64
positive_rating
0    51.822762
1    48.177238
Name: proportion, dtype: float64


In [16]:
print(model_data.columns.tolist())

['userId', 'movieId', 'rating', 'timestamp', 'rating_datetime', '(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western', 'positive_rating']


In [17]:
genre_columns = genre_features.columns.tolist()

print(genre_columns)

['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


## 5. Train/Test Strategy

What information counts as the past, and what information counts as the future?

In [18]:
model_data = model_data.sort_values(
    ["userId", "rating_datetime"]
).copy()

In [19]:
model_data["interaction_number"] = (
    model_data.groupby("userId").cumcount() + 1
)

model_data["interaction_count"] = (
    model_data.groupby("userId")["movieId"].transform("size")
)

In [20]:
model_data["is_test"] = (
    model_data["interaction_number"]
    > 0.8 * model_data["interaction_count"]
)

In [21]:
train_data = model_data[~model_data["is_test"]].copy()
test_data = model_data[model_data["is_test"]].copy()

In [22]:
print("Training rows:", train_data.shape[0])
print("Testing rows:", test_data.shape[0])

print("Training users:", train_data["userId"].nunique())
print("Testing users:", test_data["userId"].nunique())

Training rows: 80419
Testing rows: 20417
Training users: 610
Testing users: 610


In [23]:
train_last_date = (
    train_data.groupby("userId")["rating_datetime"].max()
)

test_first_date = (
    test_data.groupby("userId")["rating_datetime"].min()
)

temporal_check = train_last_date <= test_first_date

print("Users with correct chronological split:", temporal_check.sum())
print("Total users checked:", len(temporal_check))
print("Any violations:", (~temporal_check).sum())

Users with correct chronological split: 610
Total users checked: 610
Any violations: 0


## 6. Validate Temporal Train/Test Split

In [24]:
latest_train = (
    train_data.groupby("userId")["rating_datetime"]
    .max()
    .rename("latest_train_time")
)

earliest_test = (
    test_data.groupby("userId")["rating_datetime"]
    .min()
    .rename("earliest_test_time")
)

temporal_check = pd.concat(
    [latest_train, earliest_test],
    axis=1
)

temporal_check.head()

,latest_train_time,earliest_test_time
userId,,
1,2000-07-30 18:54:23+00:00,2000-07-30 18:54:42+00:00
2,2015-10-24 19:32:46+00:00,2015-10-24 19:32:52+00:00
3,2011-05-27 02:43:09+00:00,2011-05-27 02:43:24+00:00
4,2001-04-09 20:47:06+00:00,2001-04-09 20:47:18+00:00
5,1996-11-08 06:41:32+00:00,1996-11-08 06:41:32+00:00


In [25]:
print("Users checked:", len(temporal_check))

print(
    "Users missing training data:",
    temporal_check["latest_train_time"].isna().sum()
)

print(
    "Users missing testing data:",
    temporal_check["earliest_test_time"].isna().sum()
)

Users checked: 610
Users missing training data: 0
Users missing testing data: 0


In [26]:
temporal_check["temporal_violation"] = (
    temporal_check["latest_train_time"]
    > temporal_check["earliest_test_time"]
)

temporal_violations = temporal_check[
    temporal_check["temporal_violation"]
]

print("Temporal violations:", len(temporal_violations))

Temporal violations: 0


In [27]:
same_boundary_time = (
    temporal_check["latest_train_time"]
    == temporal_check["earliest_test_time"]
)

print(
    "Users with equal train/test boundary timestamps:",
    same_boundary_time.sum()
)

Users with equal train/test boundary timestamps: 130


## 7. Validate and Export Prepared Tables

Check the dimensions and column consistency of the prepared training
and test tables, then export them as CSV files in
`movielens_prepared_split.zip`.

The archive preserves the exact interaction assignments used by the
baseline and later personalized experiments. Download and retain it:
files in the Colab runtime are temporary.

CSV does not preserve all pandas data types. Downstream notebooks
must validate the loaded tables and explicitly parse datetime columns
when needed.

In [28]:
print("Training shape:", train_data.shape)
print("Testing shape:", test_data.shape)

Training shape: (80419, 29)
Testing shape: (20417, 29)


In [29]:
print(train_data.columns.tolist())

print(
    "Same columns in the same order:",
    train_data.columns.equals(test_data.columns)
)

['userId', 'movieId', 'rating', 'timestamp', 'rating_datetime', '(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western', 'positive_rating', 'interaction_number', 'interaction_count', 'is_test']
Same columns in the same order: True


In [30]:
from zipfile import ZipFile
from google.colab import files

with ZipFile("movielens_prepared_split.zip", "w") as archive:
    archive.writestr(
        "train_data.csv",
        train_data.to_csv(index=False)
    )
    archive.writestr(
        "test_data.csv",
        test_data.to_csv(index=False)
    )

files.download("movielens_prepared_split.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Preprocessing Summary

### Prepared Data

- Converted rating timestamps to UTC datetimes while retaining the
  original Unix timestamps.
- Encoded 20 genre indicators, including `(no genres listed)`.
- Joined movie genres to rating interactions.
- Defined `positive_rating`: 1 for ratings of 4.0 or higher,
  otherwise 0.

### Split Validation

| Check | Result |
|---|---:|
| Training interactions | 80,419 |
| Test interactions | 20,417 |
| Users represented in both sets | 610 |
| Users missing training or test data | 0 |
| Users with latest training time after earliest test time | 0 |
| Users with equal train/test boundary timestamps | 130 |

Equal boundary timestamps reflect the dataset's one-second timestamp
resolution. They do not establish the ordering of those tied
interactions. This ambiguity is retained and must be considered when
constructing user-history features.

### Output and Next Stage

Exported `movielens_prepared_split.zip` contains `train_data.csv`
and `test_data.csv`, each with 29 columns.

Notebook 05 selects the 20 genre inputs and evaluates the
non-personalized Logistic Regression baseline. Later experiments
must reuse these held-out test interactions for fair comparison.